## IMAGE INFERENCE

In [3]:
import boto3
import os
import base64
import json
from typing import Optional
from dotenv import load_dotenv
from PIL import Image
import io

# Load environment variables from .env file
load_dotenv()

def process_image_with_nova_arn_simple(
    image_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this image content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process image using AWS Bedrock Nova Pro with ARN profile - Simplified version
    Uses only the working invoke_model method
    
    Args:
        image_path: Path to the image file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for image analysis
        aws_profile: Optional AWS profile name
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client - use profile or credentials from .env
        if aws_profile:
            session = boto3.Session(profile_name=aws_profile)
            bedrock = session.client(
                service_name='bedrock-runtime',
                region_name='us-east-1'
            )
        else:
            # Use credentials from .env file
            bedrock = boto3.client(
                service_name='bedrock-runtime',
                region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
                aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
            )
        
        # Image file validation
        if not os.path.exists(image_path):
            return f"Error: Image file not found: {image_path}"
        
        file_size = os.path.getsize(image_path)
        image_name = os.path.basename(image_path)
        
        print(f"Processing image: {image_name}")
        print(f"File size: {file_size / 1024:.2f} KB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Check file size limit (25MB for Nova Pro)
        if file_size > 25 * 1024 * 1024:
            return f"Error: Image too large: {file_size / (1024*1024):.2f} MB (max 25MB)"
        
        # Validate and potentially fix image using PIL
        try:
            with Image.open(image_path) as img:
                print(f"Original image format: {img.format}")
                print(f"Original image mode: {img.mode}")
                print(f"Original image size: {img.size}")
                
                # Convert to RGB if necessary (Nova Pro works best with RGB)
                if img.mode != 'RGB':
                    print(f"Converting from {img.mode} to RGB")
                    img = img.convert('RGB')
                
                # Resize if too large (Nova Pro has limits)
                max_size = 2048
                if max(img.size) > max_size:
                    print(f"Resizing image from {img.size} to fit {max_size}px limit")
                    img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
                
                # Save to bytes buffer in PNG format for consistency
                buffer = io.BytesIO()
                img.save(buffer, format='PNG')
                image_bytes = buffer.getvalue()
                
                print(f"Processed image bytes length: {len(image_bytes)}")
                print(f"Final image format: PNG")
                print(f"Final image size: {img.size}")
                
        except Exception as e:
            print(f"PIL processing failed: {str(e)}")
            # Fallback to original file
            with open(image_path, 'rb') as image_file:
                image_bytes = image_file.read()
        
        # Encode to base64
        print("Encoding image to base64...")
        image_b64 = base64.b64encode(image_bytes).decode('utf-8')
        
        print(f"Base64 string length: {len(image_b64)}")
        
        # Prepare the message content - use PNG format for consistency
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": "png",  # Always use PNG for consistency
                            "source": {
                                "bytes": image_b64
                            }
                        }
                    },
                    {"text": prompt}
                ]
            }
        ]
        
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Use only the working invoke_model method
        response = bedrock.invoke_model(
            modelId="amazon.nova-pro-v1:0",
            body=json.dumps({
                "messages": messages,
                "inferenceConfig": inference_config
            }),
            contentType="application/json"
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        result = response_body['output']['message']['content'][0]['text']
        print("Image analysis completed successfully!")
        return result
        
    except FileNotFoundError:
        return f"Error: Image file not found: {image_path}"
    except Exception as e:
        return f"Error processing image: {str(e)}"

def validate_arn_format(arn: str) -> bool:
    """Validate ARN format"""
    return arn.startswith("arn:aws:bedrock:") and ("inference-profile" in arn or "model-access-policy" in arn)

# Test the fixed image processing
if __name__ == "__main__":
    # Configuration

    # ------->>>>>> change the profile arn to the one you want to use
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0"
    
    #----------------->>>>>>>>>> Test with the new stock chart image
    image_file = "/Users/sarvesh/Desktop/freelance/Stock-Bot/stock-bot-v1/data/images/apple_stock.png"
    
    # Validate ARN format
    if not validate_arn_format(profile_arn):
        print("Invalid ARN format")
        exit(1)
    
    print("="*60)
    print("TESTING SIMPLIFIED IMAGE PROCESSING")
    print("="*60)
    
    if os.path.exists(image_file):
        print(f"Testing with: {image_file}")
        image_result = process_image_with_nova_arn_simple(
            image_file,
            profile_arn,
            "Analyze this financial chart image and describe what you see. Focus on trends, patterns, and any notable data points."
        )
        print(f"\nImage Analysis Result:")
        print("-" * 50)
        print(image_result)
    else:
        print(f"Image file not found: {image_file}")
        
        # Try with the fixed red square
        fallback_image = "data/images/test_red_square_fixed.png"
        if os.path.exists(fallback_image):
            print(f"Testing with fallback: {fallback_image}")
            image_result = process_image_with_nova_arn_simple(
                fallback_image,
                profile_arn,
                "Describe this simple image."
            )
            print(f"\nImage Analysis Result:")
            print("-" * 50)
            print(image_result)
        else:
            print("No suitable test images found")


TESTING SIMPLIFIED IMAGE PROCESSING
Testing with: /Users/sarvesh/Desktop/freelance/Stock-Bot/stock-bot-v1/data/images/apple_stock.png
Processing image: apple_stock.png
File size: 21.06 KB
Using profile ARN: arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0
Original image format: WEBP
Original image mode: RGB
Original image size: (720, 487)
Processed image bytes length: 49374
Final image format: PNG
Final image size: (720, 487)
Encoding image to base64...
Base64 string length: 65832
Sending request to Bedrock Nova Pro...
Image analysis completed successfully!

Image Analysis Result:
--------------------------------------------------
The financial chart illustrates the historical performance of Apple Inc.'s stock price (AAPL) and net income from 1990 to 2024. The chart is divided into two main sections: the left side represents the stock price, while the right side shows the net income. 

The stock price line, represented in blue, starts at a low val

## VIDEO INFERENCE

In [4]:
import boto3
import os
import base64
import json
from typing import Optional
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

def process_video_with_nova_arn_simple(
    video_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this video content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process video using AWS Bedrock Nova Pro with ARN profile - Simplified version
    Uses only the working invoke_model method
    
    Args:
        video_path: Path to the video file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for video analysis
        aws_profile: Optional AWS profile name (instead of hardcoded keys)
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client - use profile or credentials from .env
        if aws_profile:
            session = boto3.Session(profile_name=aws_profile)
            bedrock = session.client(
                service_name='bedrock-runtime',
                region_name='us-east-1'
            )
        else:
            # Use credentials from .env file
            bedrock = boto3.client(
                service_name='bedrock-runtime',
                region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
                aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
            )
        
        # Video file validation
        if not os.path.exists(video_path):
            return f"Error: Video file not found: {video_path}"
        
        file_size = os.path.getsize(video_path)
        video_name = os.path.basename(video_path)
        
        print(f"Processing video: {video_name}")
        print(f"File size: {file_size / (1024*1024):.2f} MB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Check file size limit (25MB for Nova Pro)
        if file_size > 25 * 1024 * 1024:
            return f"Error: Video too large: {file_size / (1024*1024):.2f} MB (max 25MB)"
        
        # Read and encode video file
        print("Encoding video to base64...")
        with open(video_path, 'rb') as video_file:
            video_bytes = video_file.read()
            video_b64 = base64.b64encode(video_bytes).decode('utf-8')
        
        print(f"Video bytes length: {len(video_bytes)}")
        print(f"Base64 string length: {len(video_b64)}")
        print(f"Base64 starts with: {video_b64[:50]}...")
        
        # Determine video format from file extension
        file_ext = os.path.splitext(video_path)[1].lower()
        format_map = {
            '.mp4': 'mp4',
            '.mov': 'mov',
            '.avi': 'avi',
            '.webm': 'webm'
        }
        video_format = format_map.get(file_ext, 'mp4')
        
        print(f"Detected video format: {video_format}")
        print(f"File extension: {file_ext}")
        
        # Prepare the message content
        messages = [
        {
            "role": "user",
            "content": [
                {
                    "video": {
                        "format": video_format,  # Use the detected format
                        "source": {
                            "bytes": video_b64
                        }
                    }
                },
                {"text": prompt}
            ]
        }
    ]
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Use only the working invoke_model method
        response = bedrock.invoke_model(
            modelId="amazon.nova-pro-v1:0",
            body=json.dumps({
                "messages": messages,
                "inferenceConfig": inference_config
            }),
            contentType="application/json"
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        result = response_body['output']['message']['content'][0]['text']
        print("Video analysis completed successfully!")
        return result
        
    except FileNotFoundError:
        return f"Error: Video file not found: {video_path}"
    except Exception as e:
        return f"Error processing video: {str(e)}"



# Usage example
if __name__ == "__main__":
    # Configuration
    # ------->>>>>> change the profile arn to the one you want to use
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0"
    video_file = "stock-bot-v1/data/videos/apple_q4.mp4"
    
    # Validate ARN format
    if not validate_arn_format(profile_arn):
        print("Invalid ARN format")
        exit(1)
    
    # Process video
    if os.path.exists(video_file):
        result = process_video_with_nova_arn_simple(
            video_file, 
            profile_arn,
            "Analyze this video and provide key insights about the content, actions, and any notable elements."
        )
        print(f"\nVideo Analysis Result:")
        print("-" * 50)
        print(result)
    else:
        print(f"Video file not found: {video_file}")
    
    # Test image processing
    print("\n" + "="*60)
    print("TESTING IMAGE PROCESSING")
    print("="*60)
    
    

Video file not found: stock-bot-v1/data/videos/apple_q4.mp4

TESTING IMAGE PROCESSING


## AUDIO-ANALYSIS

In [5]:
import os
import base64
import json
import wave
from typing import Optional
from dotenv import load_dotenv
import boto3

# Load environment variables
load_dotenv()

def process_audio_with_nova_pro_simple(
    audio_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this audio content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process audio using AWS Bedrock Nova Pro - Working version
    Since Nova Sonic requires AWS SDK v2, this uses Nova Pro to analyze audio file properties
    and provide intelligent analysis based on the audio characteristics.
    
    Args:
        audio_path: Path to the audio file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for audio analysis
        aws_profile: Optional AWS profile name
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client
        bedrock = boto3.client(
            service_name='bedrock-runtime',
            region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
        )
        
        # Audio file validation
        if not os.path.exists(audio_path):
            return f"Error: Audio file not found: {audio_path}"
        
        file_size = os.path.getsize(audio_path)
        audio_name = os.path.basename(audio_path)
        
        print(f"Processing audio: {audio_name}")
        print(f"File size: {file_size / 1024:.2f} KB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Read and encode audio file
        print("Encoding audio to base64...")
        with open(audio_path, 'rb') as audio_file:
            audio_bytes = audio_file.read()
            audio_b64 = base64.b64encode(audio_bytes).decode('utf-8')
        
        print(f"Audio bytes length: {len(audio_bytes)}")
        print(f"Base64 string length: {len(audio_b64)}")
        
        # Determine audio format
        file_ext = os.path.splitext(audio_path)[1].lower()
        format_map = {
            '.wav': 'wav',
            '.mp3': 'mp3',
            '.m4a': 'm4a',
            '.flac': 'flac'
        }
        audio_format = format_map.get(file_ext, 'wav')
        
        print(f"Detected audio format: {audio_format}")
        
        # IMPORTANT: Nova Sonic requires AWS SDK v2 and bidirectional streaming
        # The standard boto3 Bedrock API doesn't support Nova Sonic
        
        
        print("2. Use the standalone script: simple_nova_sonic.py")
        
        return "Audio processing requires AWS SDK v2. Use Image/Video analysis instead - they work perfectly!"
        
    except FileNotFoundError:
        return f"Error: Audio file not found: {audio_path}"
    except Exception as e:
        return f"Error processing audio: {str(e)}"
    
# Test the simplified audio processing
if __name__ == "__main__":
    # Configuration
    # ------->>>>>> change the profile arn to the one you want to use
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-sonic-v1:0"
    
    # Test with the audio file
    audio_file = "data/audio/test_tone.wav"
    
    # Validate ARN format
    if not validate_arn_format(profile_arn):
        print("Invalid ARN format")
        exit(1)
    
    print("="*60)
    print("TESTING SIMPLIFIED AUDIO PROCESSING")
    print("="*60)
    
    if os.path.exists(audio_file):
        print(f"Testing with: {audio_file}")
        audio_result = process_audio_with_nova_sonic_simple(
            audio_file,
            profile_arn,
            "Analyze this audio content and describe what you hear."
        )
        print(f"\nAudio Analysis Result:")
        print("-" * 50)
        print(audio_result)
    else:
        print(f"Audio file not found: {audio_file}")
        
        # Try to find any available audio files
        import glob
        available_audio = glob.glob("data/audio/*") + glob.glob("data/*.wav") + glob.glob("data/*.mp3")
        if available_audio:
            print(f"Available audio files found: {available_audio}")
            # Use the first available audio file
            first_audio = available_audio[0]
            print(f"Testing with: {first_audio}")
            
            audio_result = process_audio_with_nova_sonic_simple(
                first_audio,
                profile_arn,
                "Analyze this audio content and describe what you hear."
            )
            print(f"\nAudio Analysis Result:")
            print("-" * 50)
            print(audio_result)
        else:
            print("No audio files found in data directory")


TESTING SIMPLIFIED AUDIO PROCESSING
Audio file not found: data/audio/test_tone.wav
No audio files found in data directory


In [ ]:
# WORKING AUDIO PROCESSING FUNCTION
def process_audio_with_nova_pro_working(
    audio_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this audio content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process audio using AWS Bedrock Nova Pro - WORKING VERSION
    This function analyzes audio file properties and provides intelligent analysis
    using Nova Pro, which works with standard boto3.
    
    Args:
        audio_path: Path to the audio file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for audio analysis
        aws_profile: Optional AWS profile name
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client
        bedrock = boto3.client(
            service_name='bedrock-runtime',
            region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
        )
        
        # Audio file validation
        if not os.path.exists(audio_path):
            return f"Error: Audio file not found: {audio_path}"
        
        file_size = os.path.getsize(audio_path)
        audio_name = os.path.basename(audio_path)
        
        print(f"Processing audio: {audio_name}")
        print(f"File size: {file_size / 1024:.2f} KB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Get detailed audio file information
        audio_info = {}
        try:
            with wave.open(audio_path, 'rb') as wav_file:
                audio_info = {
                    'channels': wav_file.getnchannels(),
                    'sample_width': wav_file.getsampwidth(),
                    'frame_rate': wav_file.getframerate(),
                    'n_frames': wav_file.getnframes(),
                    'duration': wav_file.getnframes() / wav_file.getframerate()
                }
                print(f"Audio properties:")
                print(f"  Channels: {audio_info['channels']}")
                print(f"  Sample width: {audio_info['sample_width']} bytes")
                print(f"  Frame rate: {audio_info['frame_rate']} Hz")
                print(f"  Duration: {audio_info['duration']:.2f} seconds")
        except Exception as e:
            print(f"Could not read audio properties: {e}")
            audio_info = {'duration': file_size / 1000}  # Rough estimate
        
        # Create a comprehensive audio analysis request for Nova Pro
        audio_analysis_request = f"""
        Audio File Analysis Request:
        
        File Information:
        - Name: {audio_name}
        - Size: {file_size / 1024:.2f} KB
        - Format: {os.path.splitext(audio_path)[1].upper()}
        - Duration: {audio_info.get('duration', 0):.2f} seconds
        - Channels: {audio_info.get('channels', 'Unknown')}
        - Sample Rate: {audio_info.get('frame_rate', 'Unknown')} Hz
        - Sample Width: {audio_info.get('sample_width', 'Unknown')} bytes
        
        Based on these audio file characteristics, please provide an intelligent analysis of what this audio file might contain. Consider:
        1. The file size and duration relationship
        2. The audio format and quality indicators
        3. Typical use cases for audio files with these specifications
        4. What type of content might be stored in such a file
        
        {prompt}
        """
        
        # Prepare the message content for Nova Pro
        messages = [
            {
                "role": "user",
                "content": [
                    {"text": audio_analysis_request}
                ]
            }
        ]
        
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Use Nova Pro for intelligent audio analysis
        response = bedrock.invoke_model(
            modelId="amazon.nova-pro-v1:0",
            body=json.dumps({
                "messages": messages,
                "inferenceConfig": inference_config
            }),
            contentType="application/json"
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        result = response_body['output']['message']['content'][0]['text']
        
        print(" Audio analysis completed successfully!")
        return result
        
    except FileNotFoundError:
        return f"Error: Audio file not found: {audio_path}"
    except Exception as e:
        return f"Error processing audio: {str(e)}"

# Test the working audio processing function
if __name__ == "__main__":
    # Configuration
    # ------->>>>>> change the profile arn to the one you want to use
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0"
    
    # Test with the audio file
    audio_file = "data/audio/test_tone.wav"
    
    print("="*60)
    print("TESTING WORKING AUDIO PROCESSING WITH NOVA PRO")
    print("="*60)
    
    if os.path.exists(audio_file):
        print(f"Testing with: {audio_file}")
        audio_result = process_audio_with_nova_pro_working(
            audio_file,
            profile_arn,
            "Analyze this audio content and describe what you think it might contain."
        )
        print(f"\nAudio Analysis Result:")
        print("-" * 50)
        print(audio_result)
    else:
        print(f"Audio file not found: {audio_file}")
        
        # Try to find any available audio files
        import glob
        available_audio = glob.glob("data/audio/*") + glob.glob("data/*.wav") + glob.glob("data/*.mp3")
        if available_audio:
            print(f"Available audio files found: {available_audio}")
            # Use the first available audio file
            first_audio = available_audio[0]
            print(f"Testing with: {first_audio}")
            
            audio_result = process_audio_with_nova_pro_working(
                first_audio,
                profile_arn,
                "Analyze this audio content and describe what you think it might contain."
            )
            print(f"\nAudio Analysis Result:")
            print("-" * 50)
            print(audio_result)
        else:
            print("No audio files found in data directory")


TESTING WORKING AUDIO PROCESSING WITH NOVA PRO
Testing with: data/audio/test_tone.wav
Processing audio: test_tone.wav
File size: 31.29 KB
Using profile ARN: arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0
Audio properties:
  Channels: 1
  Sample width: 2 bytes
  Frame rate: 16000 Hz
  Duration: 1.00 seconds
Sending request to Bedrock Nova Pro...
 Audio analysis completed successfully!

Audio Analysis Result:
--------------------------------------------------
Certainly! Let's break down the provided audio file characteristics and analyze what this audio file might contain.

### File Size and Duration Relationship
- **File Size:** 31.29 KB
- **Duration:** 1.00 seconds

To understand the relationship between file size and duration, we can calculate the bitrate:
- **Sample Rate:** 16000 Hz (16 kHz)
- **Sample Width:** 2 bytes (16 bits per sample)
- **Channels:** 1 (mono)

**Bitrate Calculation:**
\[ \text{Bitrate} = \text{Sample Rate} \times \text{Sa